In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)

print(answer)

The agentic loop keeps calling the model until it stops by utilizing a `while` loop that continually checks if the model has requested any function calls. The key points are:

1. **Iteration Tracking**: Each iteration of the loop represents a new call to the model. An iteration counter is used to keep track of how many times the loop has run.

2. **Function Call Check**: After each response from the model, the loop examines whether the response includes any function calls. If it does, the loop will continue; if it does not, the loop will exit.

3. **Response Processing**: The loop processes the model’s output, appends it to the conversation history, and checks if there are any function calls. If a function call is detected, it executes the call and sends the result back to the model in the next iteration.

4. **Exit Condition**: The loop terminates when the model provides a response that does not include further function calls, indicating that it has all the information it needs to pro

In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [3]:
import starter
from rag_helper import RAGBase

In [4]:
print("Model:", starter.rag.model)
print("Base URL:", starter.rag.llm_client.base_url)

Model: openai/gpt-4o-mini
Base URL: https://models.github.ai/inference/


In [5]:
class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            return response

In [6]:
traced_rag = RAGTraced(
    index=starter.index,
    llm_client=starter.client,
    model="openai/gpt-4o-mini"
)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)

print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x703f9bf8c2348b4d7688ce7d2a9a984b",
        "span_id": "0x1c4a7baf76058a76",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb257a38fc6d6eae8",
    "start_time": "2026-07-19T19:23:10.360207Z",
    "end_time": "2026-07-19T19:23:10.362676Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "eb6fbbd1-21c5-4c34-97e0-a5e3e49631af",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x703f9bf8c2348b4d7688ce7d2a9a984b",
        "span_id": "0x96ed936d926c2a20",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [8]:
class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            input_tokens = usage.prompt_tokens
            output_tokens = usage.completion_tokens

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            input_price_per_1m = 0.15
            output_price_per_1m = 0.60

            cost = (
                input_tokens / 1_000_000 * input_price_per_1m
                + output_tokens / 1_000_000 * output_price_per_1m
            )

            span.set_attribute("cost", cost)

            return response

In [9]:
traced_rag = RAGTraced(
    index=starter.index,
    llm_client=starter.client,
    model="openai/gpt-4o-mini"
)

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)

print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x59cff252c5172b604f0b53e64df0ebeb",
        "span_id": "0xcfec037b154590b1",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x80ffed0c41ce7d5c",
    "start_time": "2026-07-19T19:27:28.616992Z",
    "end_time": "2026-07-19T19:27:28.619334Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "eb6fbbd1-21c5-4c34-97e0-a5e3e49631af",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x59cff252c5172b604f0b53e64df0ebeb",
        "span_id": "0x19781e2d3344c2e8",
        "trace_state": "[]"
    },
    "kind": "SpanKind

In [11]:
from datetime import datetime

llm_start = "2026-07-19T19:23:10.366444Z"
llm_end = "2026-07-19T19:23:13.108112Z"

start_dt = datetime.fromisoformat(llm_start.replace("Z", "+00:00"))
end_dt = datetime.fromisoformat(llm_end.replace("Z", "+00:00"))

duration_seconds = (end_dt - start_dt).total_seconds()
duration_ms = duration_seconds * 1000

print("LLM duration seconds:", duration_seconds)
print("LLM duration ms:", duration_ms)

if duration_ms < 100:
    answer = "Under 100ms"
elif duration_ms < 500:
    answer = "100-500ms"
elif duration_ms < 2000:
    answer = "500-2000ms"
else:
    answer = "Over 2000ms"

print("Q3 answer:", answer)

LLM duration seconds: 2.741668
LLM duration ms: 2741.668
Q3 answer: Over 2000ms


In [1]:
import os
import sqlite3

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult, SimpleSpanProcessor


# Optional: start fresh for Q4/Q5
if os.path.exists("traces.db"):
    os.remove("traces.db")


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})

            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )

        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self, timeout_millis=30000):
        self.conn.close()

    def force_flush(self, timeout_millis=30000):
        return True


provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
import starter
from rag_helper import RAGBase

print("Model:", starter.rag.model)
print("Base URL:", starter.rag.llm_client.base_url)

Model: openai/gpt-4o-mini
Base URL: https://models.github.ai/inference/


In [3]:
class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            input_tokens = usage.prompt_tokens
            output_tokens = usage.completion_tokens

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            input_price_per_1m = 0.15
            output_price_per_1m = 0.60

            cost = (
                input_tokens / 1_000_000 * input_price_per_1m
                + output_tokens / 1_000_000 * output_price_per_1m
            )

            span.set_attribute("cost", cost)

            return response

In [4]:
traced_rag = RAGTraced(
    index=starter.index,
    llm_client=starter.client,
    model="openai/gpt-4o-mini"
)

In [5]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = traced_rag.rag(query)

print(answer)

The agentic loop keeps calling the model until it stops by using a `while` loop that checks for function calls. In the loop, each iteration sends the current state of messages to the model. The model receives these messages and generates a response. If the response includes a function call (indicating that the model wants to take additional action, like searching for more information), the loop executes that function and appends the output to the message history. The loop continues iterating in this manner until the model returns a response without any function calls, at which point it breaks out of the loop and concludes the process. The exit condition is simply that no function calls are present in the latest model response.


In [6]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

df = pd.read_sql_query("SELECT * FROM spans", conn)

df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784489647549760592,1784489647552358437,NaN,NaN,NaN
1,llm,1784489647558618336,1784489651657972643,7112.0,141.0,0.001151
2,rag,1784489647549700920,1784489651660998414,NaN,NaN,NaN


In [7]:
df["name"].value_counts()

name
search    1
llm       1
rag       1
Name: count, dtype: int64

In [9]:
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000

df[["name", "duration_ms", "input_tokens", "output_tokens", "cost"]]

,name,duration_ms,input_tokens,output_tokens,cost
0,search,2.597845,NaN,NaN,NaN
1,llm,4099.354307,7112.0,141.0,0.001151
2,rag,4111.297494,NaN,NaN,NaN


In [10]:
child_spans = df[df["name"] != "rag"]

duration_summary = (
    child_spans
    .groupby("name")["duration_ms"]
    .sum()
    .sort_values(ascending=False)
)

duration_summary

name
llm       4099.354307
search       2.597845
Name: duration_ms, dtype: float64

In [11]:
df["name"].value_counts()

name
search    1
llm       1
rag       1
Name: count, dtype: int64

In [12]:
df["name"].value_counts()

name
search    1
llm       1
rag       1
Name: count, dtype: int64

In [13]:
duration_summary

name
llm       4099.354307
search       2.597845
Name: duration_ms, dtype: float64

In [15]:
query = "How does the agentic loop keep calling the model until it stops?"

for i in range(3):
    answer = traced_rag.rag(query)
    print(f"Additional run {i + 1} complete")

Additional run 1 complete
Additional run 2 complete
Additional run 3 complete


In [16]:
df = pd.read_sql_query("SELECT * FROM spans", conn)

print(df["name"].value_counts())

llm_df = df[df["name"] == "llm"].copy()

print("LLM span count:", len(llm_df))
llm_df[["input_tokens", "output_tokens", "cost"]]

name
search    4
llm       4
rag       4
Name: count, dtype: int64
LLM span count: 4


,input_tokens,output_tokens,cost
1,7112.0,141.0,0.001151
4,7112.0,181.0,0.001175
7,7112.0,229.0,0.001204
10,7112.0,124.0,0.001141


In [17]:
df = pd.read_sql_query("SELECT * FROM spans", conn)

print(df["name"].value_counts())

llm_df = df[df["name"] == "llm"].copy()

llm_df[["input_tokens", "output_tokens", "cost"]]

name
search    4
llm       4
rag       4
Name: count, dtype: int64


,input_tokens,output_tokens,cost
1,7112.0,141.0,0.001151
4,7112.0,181.0,0.001175
7,7112.0,229.0,0.001204
10,7112.0,124.0,0.001141


In [18]:
tokens = llm_df["input_tokens"].dropna()

print("Input tokens:", tokens.tolist())
print("min:", tokens.min())
print("max:", tokens.max())
print("mean:", tokens.mean())

variation = (tokens.max() - tokens.min()) / tokens.mean()

print("variation:", variation)
print("variation percent:", variation * 100)

Input tokens: [7112.0, 7112.0, 7112.0, 7112.0]
min: 7112.0
max: 7112.0
mean: 7112.0
variation: 0.0
variation percent: 0.0
